In [1]:
import torch

from transformers import AutoTokenizer

#加载tokenizer
tokenizer = AutoTokenizer.from_pretrained(r'C:\Users\ssw\Desktop\CS_Code\Machine_learn\ner\model\bert-base-chinese')

tokenizer

C:\Users\ssw\.conda\envs\ner\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


BertTokenizerFast(name_or_path='C:\Users\ssw\Desktop\CS_Code\Machine_learn\ner\model\bert-base-chinese', vocab_size=21128, model_max_length=512, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, clean_up_tokenization_spaces=True),  added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}

In [2]:
from datasets import load_dataset

#加载数据集
dataset = load_dataset(path='lansinuote/ChnSentiCorp')

dataset, dataset['train'][0]

(DatasetDict({
     train: Dataset({
         features: ['text', 'label'],
         num_rows: 9600
     })
     validation: Dataset({
         features: ['text', 'label'],
         num_rows: 1200
     })
     test: Dataset({
         features: ['text', 'label'],
         num_rows: 1200
     })
 }),
 {'text': '选择珠江花园的原因就是方便，有电动扶梯直接到达海边，周围餐馆、食廊、商场、超市、摊位一应俱全。酒店装修一般，但还算整洁。 泳池在大堂的屋顶，因此很小，不过女儿倒是喜欢。 包的早餐是西式的，还算丰富。 服务吗，一般',
  'label': 1})

In [4]:
#定义数据集遍历工具
def collate_fn(data):
    text = [i['text'] for i in data]
    label = [i['label'] for i in data]

    #文字编码
    data = tokenizer(text,
                     padding=True,
                     truncation=True,
                     max_length=500,
                     return_tensors='pt',
                     return_token_type_ids=False)

    #设置label
    data['label'] = torch.LongTensor(label)

    return data


loader = torch.utils.data.DataLoader(dataset['train'],
                                     batch_size=8,
                                     shuffle=True,
                                     drop_last=True,
                                     collate_fn=collate_fn)

data = next(iter(loader))

for k, v in data.items():
    print(k, v.shape)

len(loader)

input_ids torch.Size([8, 142])
attention_mask torch.Size([8, 142])
label torch.Size([8])


1200

In [12]:
#定义模型
class Model(torch.nn.Module):

    def __init__(self):
        super().__init__()

        #加载预训练模型
        from transformers import AutoModel
        self.pretrained = AutoModel.from_pretrained(
            r'C:\Users\ssw\Desktop\CS_Code\Machine_learn\ner\model\bert-base-chinese')

        self.fc = torch.nn.Linear(in_features=768, out_features=2)

    def forward(self, input_ids, attention_mask, label=None):
        #使用预训练模型抽取数据特征
        with torch.no_grad():
            last_hidden_state = self.pretrained(
                input_ids=input_ids,
                attention_mask=attention_mask).last_hidden_state

        #只取第0个词的特征做分类,这和bert模型的训练方式有关,此处不展开
        last_hidden_state = last_hidden_state[:, 0]

        #对抽取的特征只取第一个字的结果做分类即可
        out = self.fc(last_hidden_state).softmax(dim=1)

        #计算loss
        loss = None
        if label is not None:
            loss = torch.nn.functional.cross_entropy(out, label)

        return loss, out


model = Model()

model(**data)# 打印loss如下,可简单认为八个句子,进行二分类,两者概率相加是 = 1 的

(tensor(0.7026, grad_fn=<NllLossBackward0>),
 tensor([[0.5868, 0.4132],
         [0.4845, 0.5155],
         [0.6314, 0.3686],
         [0.5920, 0.4080],
         [0.6653, 0.3347],
         [0.4493, 0.5507],
         [0.6463, 0.3537],
         [0.4353, 0.5647]], grad_fn=<SoftmaxBackward0>))

In [13]:
#执行训练
#将上面拿到的数据的特征loss进行训练
def train():
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
    #
    for i, data in enumerate(loader):
        loss, out = model(**data)

        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if i % 10 == 0:
            out = out.argmax(dim=1)
            acc = (out == data.label).sum().item() / len(data.label)
            print(i, len(loader), loss.item(), acc)
            # 批次 总数 loss 正确率(0.5相当于是瞎猜的)

        if i == 300:
            break


train()

0 1200 0.7203945517539978 0.375
10 1200 0.6802307367324829 0.75
20 1200 0.7224258184432983 0.375
30 1200 0.7085728645324707 0.625
40 1200 0.7033324837684631 0.5
50 1200 0.6480254530906677 0.625
60 1200 0.6388052701950073 0.75
70 1200 0.6235789060592651 0.875
80 1200 0.6787087321281433 0.625
90 1200 0.6580554246902466 0.75
100 1200 0.6022767424583435 0.875
110 1200 0.6374906897544861 0.875
120 1200 0.6637921333312988 0.5
130 1200 0.626818835735321 0.75
140 1200 0.5878689289093018 1.0
150 1200 0.665015697479248 0.625
160 1200 0.6805894374847412 0.5
170 1200 0.5561002492904663 1.0
180 1200 0.6307578086853027 0.75
190 1200 0.5538347959518433 1.0
200 1200 0.5733105540275574 0.875
210 1200 0.6117584109306335 0.625
220 1200 0.5088293552398682 1.0
230 1200 0.5703542232513428 1.0
240 1200 0.5606151819229126 0.875
250 1200 0.537761390209198 0.875
260 1200 0.5385181903839111 0.75
270 1200 0.5048703551292419 1.0
280 1200 0.527568519115448 1.0
290 1200 0.5211682319641113 0.875
300 1200 0.5703141093

In [14]:
#执行测试
def test():
    loader_test = torch.utils.data.DataLoader(dataset['test'],
                                              batch_size=8,
                                              shuffle=True,
                                              drop_last=True,
                                              collate_fn=collate_fn)

    correct = 0
    total = 0
    for i, data in enumerate(loader_test):
        with torch.no_grad():
            _, out = model(**data)

        out = out.argmax(dim=1)
        correct += (out == data.label).sum().item()
        total += len(data.label)

        print(i, len(loader_test), correct / total)

        if i == 5:
            break

    return correct / total


test()

0 150 0.75
1 150 0.875
2 150 0.9166666666666666
3 150 0.9375
4 150 0.95
5 150 0.9375


0.9375